# Benchmark JAX GPU vs Python Numba JIT — Geosteering AI v1.6.0

## Objetivo

Comparar o desempenho do simulador de resistividade EM 1D em dois backends:

| Backend | Implementação | Dispositivo |
|---------|--------------|-------------|
| **Numba JIT** | Python + Numba `@njit(cache=True)` | CPU |
| **JAX GPU** | JAX `jit` + XLA unified (Sprint 10) | GPU T4 / CPU fallback |

## Configurações de Benchmark

| Config | Frequências | Pares TR | Ângulos | Posições |
|--------|-------------|----------|---------|----------|
| **A** | 1 × 20 kHz | 1 × 1 m | 1 × 0° | 600 pts |
| **B** | 2 × (20 kHz, 40 kHz) | 1 × 1 m | 2 × (0°, 30°) | 600 pts |

## Modelos Canônicos Testados

Oklahoma 3, Oklahoma 5, Oklahoma 28, Devine 8, Hou 7, Viking Graben 10

## Experimento Adicional

Geração de **30.000 modelos distintos** (Config A): 20 camadas, alta resistividade (> 1000 Ω·m),  
perfis aleatórios com semente reprodutível. Mede throughput absoluto de produção.

---

> **Ambiente alvo:** Google Colab Pro+ T4 (15 GB VRAM) | Runtime: Python 3.10 | JAX 0.4.30+  
> **Tempo estimado:** ~20–35 min (depende do dispositivo disponível)


In [ ]:
# ── Célula 1: Instalação do pacote ───────────────────────────────────────────
import subprocess, sys

# --no-deps: instala apenas geosteering_ai sem reinstalar numpy/scipy/etc.
# Evita conflitos de versão com o ambiente Colab pré-configurado.
print("📦 Instalando geosteering_ai (sem alterar dependências do Colab)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps", "--force-reinstall",
    "git+https://github.com/daniel-guitarplayer-8/geosteering-ai.git@main"
])

from geosteering_ai.simulation import __version__ as SIM_VERSION
print(f"✅ geosteering_ai.simulation v{SIM_VERSION} pronto")

# Verifica que os filtros Hankel estão acessíveis
from geosteering_ai.simulation.filters.loader import FilterLoader
try:
    FilterLoader().load("werthmuller_201pt")
    print("✅ Filtros Hankel (.npz) acessíveis")
except FileNotFoundError as e:
    print(f"❌ {e}")


In [ ]:
# ── Célula 2: Imports e configuração global ───────────────────────────────────
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from typing import Optional, Dict, List, Tuple
from dataclasses import dataclass

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})

# ── Imports JAX ──────────────────────────────────────────────────────────────
try:
    import jax
    import jax.numpy as jnp
    # Habilita float64 (obrigatório para paridade com Numba complex128)
    jax.config.update("jax_enable_x64", True)
    HAS_JAX = True
except ImportError:
    HAS_JAX = False
    print("⚠️  JAX não disponível — benchmarks JAX serão ignorados.")

# ── Imports Geosteering AI ───────────────────────────────────────────────────
from geosteering_ai.simulation import (
    simulate,
    SimulationConfig,
)
from geosteering_ai.simulation.validation import get_canonical_model

# simulate_multi (Numba) — Config B multi-freq/multi-dip
try:
    from geosteering_ai.simulation import simulate_multi as simulate_multi_numba
except ImportError:
    from geosteering_ai.simulation.multi_forward import simulate_multi as simulate_multi_numba

# simulate_multi_jax (JAX) — ambas as configs
if HAS_JAX:
    try:
        from geosteering_ai.simulation import simulate_multi_jax
    except ImportError:
        from geosteering_ai.simulation._jax.multi_forward import simulate_multi_jax
    from geosteering_ai.simulation._jax.forward_pure import (
        clear_unified_jit_cache,
        count_compiled_xla_programs,
    )

print("✅ Todos os imports realizados com sucesso")


In [ ]:
# ── Célula 3: Detecção de dispositivo e parâmetros de benchmark ──────────────

# Detectar dispositivo JAX disponível
if HAS_JAX:
    JAX_BACKEND = jax.default_backend()  # 'gpu' ou 'cpu'
    JAX_DEVICES = jax.devices()
    print(f"🖥️  Backend JAX padrão : {JAX_BACKEND.upper()}")
    print(f"🖥️  Dispositivos JAX   : {JAX_DEVICES}")
    if JAX_BACKEND == 'gpu':
        # Tentar obter informações da GPU
        try:
            import subprocess
            gpu_info = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]
            ).decode().strip()
            print(f"🎯 GPU disponível     : {gpu_info}")
        except Exception:
            print("🎯 GPU disponível (detalhes indisponíveis)")
    else:
        print("⚠️  GPU não detectada — benchmark JAX rodará em CPU")
else:
    JAX_BACKEND = None

# ── Configurações dos SimulationConfig ───────────────────────────────────────
CFG_NUMBA_A = SimulationConfig(
    backend="numba",
    frequency_hz=20000.0,
    tr_spacing_m=1.0,
    device="cpu",
)

CFG_JAX_UNIFIED = SimulationConfig(
    backend="jax",
    jax_strategy="unified",   # Sprint 10: 1 XLA program, 44→1 consolidação
    device="gpu" if (HAS_JAX and JAX_BACKEND == "gpu") else "cpu",
)

# Parâmetros de benchmark
N_WARMUP  = 3   # Chamadas de aquecimento JIT (descartadas no timer)
N_REPEATS = 7   # Repetições cronometradas (usa mediana para robustez)
N_POSITIONS = 600  # Pontos de medição ao longo do poço

# Configurações dos experimentos
CONFIG_A = dict(
    frequencies_hz=[20_000.0],
    tr_spacings_m=[1.0],
    dip_degs=[0.0],
)
CONFIG_B = dict(
    frequencies_hz=[20_000.0, 40_000.0],
    tr_spacings_m=[1.0],
    dip_degs=[0.0, 30.0],
)

print(f"\n📊 Parâmetros de benchmark:")
print(f"   N_WARMUP  = {N_WARMUP}  (chamadas JIT descartadas)")
print(f"   N_REPEATS = {N_REPEATS}  (mediana das cronometragens)")
print(f"   N_POSITIONS = {N_POSITIONS} pontos de profundidade")
print(f"   Config A: {CONFIG_A}")
print(f"   Config B: {CONFIG_B}")


In [ ]:
# ── Célula 4: Funções auxiliares de benchmark ─────────────────────────────────

@dataclass
class BenchResult:
    """Resultado de benchmark de um par (modelo, config, backend)."""
    model_name:     str
    config_name:    str
    backend:        str          # 'numba' ou 'jax'
    n_layers:       int
    n_positions:    int
    n_freqs:        int
    n_tr:           int
    n_dips:         int
    time_mean_s:    float        # Tempo médio por execução (s)
    time_median_s:  float        # Mediana dos tempos (s)
    time_std_s:     float        # Desvio padrão (s)
    throughput_mh:  float        # Modelos/hora
    speedup:        float = 1.0  # vs Numba (preenchido depois)
    xla_count:      int   = 0    # Número de programas XLA compilados (JAX)


def benchmark_numba_A(
    model_name: str,
    n_warmup: int = N_WARMUP,
    n_repeats: int = N_REPEATS,
) -> BenchResult:
    """Benchmarka Config A (1 freq, 1 TR, 0° dip) via Numba JIT."""
    m = get_canonical_model(model_name)
    positions_z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, N_POSITIONS)

    # Aquecimento JIT
    for _ in range(n_warmup):
        simulate(m.rho_h, m.rho_v, m.esp, positions_z, cfg=CFG_NUMBA_A)

    # Cronometragem
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        simulate(m.rho_h, m.rho_v, m.esp, positions_z, cfg=CFG_NUMBA_A)
        times.append(time.perf_counter() - t0)

    times = np.array(times)
    t_median = float(np.median(times))
    return BenchResult(
        model_name=model_name,
        config_name="Config A",
        backend="numba",
        n_layers=m.n_layers,
        n_positions=N_POSITIONS,
        n_freqs=1, n_tr=1, n_dips=1,
        time_mean_s=float(np.mean(times)),
        time_median_s=t_median,
        time_std_s=float(np.std(times)),
        throughput_mh=3600.0 / t_median,
    )


def benchmark_numba_B(
    model_name: str,
    n_warmup: int = N_WARMUP,
    n_repeats: int = N_REPEATS,
) -> BenchResult:
    """Benchmarka Config B (2 freqs, 1 TR, 2 dips) via Numba JIT."""
    m = get_canonical_model(model_name)
    positions_z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, N_POSITIONS)

    for _ in range(n_warmup):
        simulate_multi_numba(
            m.rho_h, m.rho_v, m.esp, positions_z,
            **CONFIG_B, cfg=CFG_NUMBA_A,
        )

    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        simulate_multi_numba(
            m.rho_h, m.rho_v, m.esp, positions_z,
            **CONFIG_B, cfg=CFG_NUMBA_A,
        )
        times.append(time.perf_counter() - t0)

    times = np.array(times)
    t_median = float(np.median(times))
    return BenchResult(
        model_name=model_name,
        config_name="Config B",
        backend="numba",
        n_layers=m.n_layers,
        n_positions=N_POSITIONS,
        n_freqs=2, n_tr=1, n_dips=2,
        time_mean_s=float(np.mean(times)),
        time_median_s=t_median,
        time_std_s=float(np.std(times)),
        throughput_mh=3600.0 / t_median,
    )


def benchmark_jax_A(
    model_name: str,
    n_warmup: int = N_WARMUP,
    n_repeats: int = N_REPEATS,
) -> BenchResult:
    """Benchmarka Config A (1 freq, 1 TR, 0° dip) via JAX unified."""
    m = get_canonical_model(model_name)
    positions_z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, N_POSITIONS)

    # Aquecimento: inclui compilação XLA (primeira chamada)
    for _ in range(n_warmup):
        res = simulate_multi_jax(
            m.rho_h, m.rho_v, m.esp, positions_z,
            **CONFIG_A, cfg=CFG_JAX_UNIFIED,
        )
        jax.block_until_ready(res.H_tensor)  # Aguarda GPU terminar

    xla_count = count_compiled_xla_programs()

    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        res = simulate_multi_jax(
            m.rho_h, m.rho_v, m.esp, positions_z,
            **CONFIG_A, cfg=CFG_JAX_UNIFIED,
        )
        jax.block_until_ready(res.H_tensor)  # ESSENCIAL para timing GPU preciso
        times.append(time.perf_counter() - t0)

    times = np.array(times)
    t_median = float(np.median(times))
    return BenchResult(
        model_name=model_name,
        config_name="Config A",
        backend=f"jax_{JAX_BACKEND}" if JAX_BACKEND else "jax_cpu",
        n_layers=m.n_layers,
        n_positions=N_POSITIONS,
        n_freqs=1, n_tr=1, n_dips=1,
        time_mean_s=float(np.mean(times)),
        time_median_s=t_median,
        time_std_s=float(np.std(times)),
        throughput_mh=3600.0 / t_median,
        xla_count=xla_count,
    )


def benchmark_jax_B(
    model_name: str,
    n_warmup: int = N_WARMUP,
    n_repeats: int = N_REPEATS,
) -> BenchResult:
    """Benchmarka Config B (2 freqs, 1 TR, 2 dips) via JAX unified."""
    m = get_canonical_model(model_name)
    positions_z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, N_POSITIONS)

    for _ in range(n_warmup):
        res = simulate_multi_jax(
            m.rho_h, m.rho_v, m.esp, positions_z,
            **CONFIG_B, cfg=CFG_JAX_UNIFIED,
        )
        jax.block_until_ready(res.H_tensor)

    xla_count = count_compiled_xla_programs()

    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        res = simulate_multi_jax(
            m.rho_h, m.rho_v, m.esp, positions_z,
            **CONFIG_B, cfg=CFG_JAX_UNIFIED,
        )
        jax.block_until_ready(res.H_tensor)
        times.append(time.perf_counter() - t0)

    times = np.array(times)
    t_median = float(np.median(times))
    return BenchResult(
        model_name=model_name,
        config_name="Config B",
        backend=f"jax_{JAX_BACKEND}" if JAX_BACKEND else "jax_cpu",
        n_layers=m.n_layers,
        n_positions=N_POSITIONS,
        n_freqs=2, n_tr=1, n_dips=2,
        time_mean_s=float(np.mean(times)),
        time_median_s=t_median,
        time_std_s=float(np.std(times)),
        throughput_mh=3600.0 / t_median,
        xla_count=xla_count,
    )


print("✅ Funções de benchmark definidas")


---
## Seção 1 — Config A: 1 Frequência, 1 TR, 1 Ângulo (0°), 600 Posições

Configuração de poço vertical padrão — benchmark de throughput base.


In [ ]:
# ── Célula 5: Benchmark Config A ─────────────────────────────────────────────
CANONICAL_MODELS = [
    "oklahoma_3",
    "oklahoma_5",
    "oklahoma_28",
    "devine_8",
    "hou_7",
    "viking_graben_10",
]

results_A: List[BenchResult] = []

print("🔄 Executando Config A — Numba JIT...")
print("-" * 60)

for model_name in CANONICAL_MODELS:
    m = get_canonical_model(model_name)
    br = benchmark_numba_A(model_name)
    results_A.append(br)
    print(
        f"  [{model_name:<20}] n={m.n_layers:>2} camadas | "
        f"mediana={br.time_median_s*1e3:>7.2f} ms | "
        f"throughput={br.throughput_mh:>9.0f} mod/h"
    )

if HAS_JAX:
    print(f"\n🔄 Executando Config A — JAX {JAX_BACKEND.upper()}...")
    print("-" * 60)
    # Limpa cache entre modelos para evitar efeitos de aquecimento cruzado
    clear_unified_jit_cache()

    for model_name in CANONICAL_MODELS:
        m = get_canonical_model(model_name)
        br = benchmark_jax_A(model_name)
        # Calcula speedup em relação ao Numba do mesmo modelo
        numba_time = next(
            r.time_median_s for r in results_A
            if r.model_name == model_name and r.backend == "numba"
        )
        br.speedup = numba_time / br.time_median_s
        results_A.append(br)
        print(
            f"  [{model_name:<20}] n={m.n_layers:>2} camadas | "
            f"mediana={br.time_median_s*1e3:>7.2f} ms | "
            f"throughput={br.throughput_mh:>9.0f} mod/h | "
            f"speedup={br.speedup:>5.2f}× | "
            f"XLA={br.xla_count}"
        )

print("\n✅ Config A concluída")


In [ ]:
# ── Célula 6: Tabela de resultados Config A ───────────────────────────────────

df_A_numba = pd.DataFrame([
    {
        "Modelo": r.model_name,
        "Camadas": r.n_layers,
        "Backend": r.backend,
        "Mediana (ms)": round(r.time_median_s * 1e3, 3),
        "Desvio (ms)": round(r.time_std_s * 1e3, 3),
        "Throughput (mod/h)": int(r.throughput_mh),
        "Throughput (mod/s)": round(r.throughput_mh / 3600, 2),
        "Speedup vs Numba": round(r.speedup, 2),
        "XLA Programs": r.xla_count,
    }
    for r in results_A
])

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
print("\n📊 Config A — 1 freq (20 kHz) | 1 TR (1 m) | 1 dip (0°) | 600 posições")
print("=" * 95)
print(df_A_numba.to_string(index=False))


---
## Seção 2 — Config B: 2 Frequências, 1 TR, 2 Ângulos (0° e 30°), 600 Posições

Configuração multi-frequência e multi-ângulo — representa poços desviados com LWD multi-canal.


In [ ]:
# ── Célula 7: Benchmark Config B ─────────────────────────────────────────────
results_B: List[BenchResult] = []

print("🔄 Executando Config B — Numba JIT...")
print("-" * 60)

for model_name in CANONICAL_MODELS:
    m = get_canonical_model(model_name)
    br = benchmark_numba_B(model_name)
    results_B.append(br)
    print(
        f"  [{model_name:<20}] n={m.n_layers:>2} camadas | "
        f"mediana={br.time_median_s*1e3:>8.2f} ms | "
        f"throughput={br.throughput_mh:>9.0f} mod/h"
    )

if HAS_JAX:
    print(f"\n🔄 Executando Config B — JAX {JAX_BACKEND.upper()}...")
    print("-" * 60)
    clear_unified_jit_cache()

    for model_name in CANONICAL_MODELS:
        m = get_canonical_model(model_name)
        br = benchmark_jax_B(model_name)
        numba_time = next(
            r.time_median_s for r in results_B
            if r.model_name == model_name and r.backend == "numba"
        )
        br.speedup = numba_time / br.time_median_s
        results_B.append(br)
        print(
            f"  [{model_name:<20}] n={m.n_layers:>2} camadas | "
            f"mediana={br.time_median_s*1e3:>8.2f} ms | "
            f"throughput={br.throughput_mh:>9.0f} mod/h | "
            f"speedup={br.speedup:>5.2f}× | "
            f"XLA={br.xla_count}"
        )

print("\n✅ Config B concluída")


In [ ]:
# ── Célula 8: Tabela de resultados Config B ───────────────────────────────────

df_B = pd.DataFrame([
    {
        "Modelo": r.model_name,
        "Camadas": r.n_layers,
        "Backend": r.backend,
        "Mediana (ms)": round(r.time_median_s * 1e3, 3),
        "Desvio (ms)": round(r.time_std_s * 1e3, 3),
        "Throughput (mod/h)": int(r.throughput_mh),
        "Throughput (mod/s)": round(r.throughput_mh / 3600, 2),
        "Speedup vs Numba": round(r.speedup, 2),
        "XLA Programs": r.xla_count,
    }
    for r in results_B
])

print("\n📊 Config B — 2 freqs (20/40 kHz) | 1 TR (1 m) | 2 dips (0°, 30°) | 600 posições")
print("=" * 100)
print(df_B.to_string(index=False))


---
## Seção 3 — Tabela Resumo Consolidada (Configs A + B)


In [ ]:
# ── Célula 9: Tabela resumo + gráfico de speedup ─────────────────────────────

all_results = results_A + results_B
df_all = pd.DataFrame([
    {
        "Modelo": r.model_name,
        "Config": r.config_name,
        "Camadas": r.n_layers,
        "Backend": r.backend,
        "Mediana (ms)": round(r.time_median_s * 1e3, 2),
        "Throughput (mod/h)": int(r.throughput_mh),
        "ms/modelo": round(r.time_median_s * 1e3, 2),
        "Speedup vs Numba": round(r.speedup, 2),
    }
    for r in all_results
])

# Pivot: comparação lado-a-lado por modelo + config
if HAS_JAX:
    df_pivot = df_all.pivot_table(
        index=["Modelo", "Config", "Camadas"],
        columns="Backend",
        values=["Mediana (ms)", "Throughput (mod/h)", "Speedup vs Numba"],
        aggfunc="first",
    ).round(2)
    print("\n📊 Tabela Resumo Consolidada — Configs A + B")
    print("=" * 110)
    print(df_pivot.to_string())

# Gráfico de barras: Speedup JAX vs Numba por modelo + config
if HAS_JAX:
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D

    jax_results = [r for r in all_results if r.backend.startswith("jax")]
    models_labels = [f"{r.model_name}\n({r.config_name})" for r in jax_results]
    speedups = [r.speedup for r in jax_results]
    colors = ["#2196F3" if r.config_name == "Config A" else "#4CAF50" for r in jax_results]

    fig, ax = plt.subplots(figsize=(14, 5))
    bars = ax.bar(range(len(jax_results)), speedups, color=colors, edgecolor="white", linewidth=0.5)
    ax.axhline(1.0, color="red",    linestyle="--", linewidth=1.2)
    ax.axhline(2.0, color="orange", linestyle=":",  linewidth=1.0)

    for bar, spd in zip(bars, speedups):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            f"{spd:.2f}×",
            ha="center", va="bottom", fontsize=8, fontweight="bold"
        )

    ax.set_xticks(range(len(jax_results)))
    ax.set_xticklabels(models_labels, fontsize=8)
    ax.set_ylabel("Speedup JAX / Numba")
    ax.set_title(
        f"Speedup JAX {JAX_BACKEND.upper()} vs Numba JIT — Geosteering AI v1.6.0\n"
        f"(Azul: Config A | Verde: Config B)"
    )
    legend_els = [
        Patch(facecolor="#2196F3", label="Config A (1f, 1TR, 0°)"),
        Patch(facecolor="#4CAF50", label="Config B (2f, 1TR, 0°+30°)"),
        Line2D([0], [0], color="red",    linestyle="--", lw=1.2, label="Baseline Numba = 1.0×"),
        Line2D([0], [0], color="orange", linestyle=":",  lw=1.0, label="Meta mínima = 2.0×"),
    ]
    ax.legend(handles=legend_els, loc="upper right", fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


---
## Seção 4 — Experimento Adicional: 30.000 Modelos Distintos (Config A)

Geração de 30.000 modelos geológicos aleatórios com:
- **20 camadas** cada
- **Alta resistividade**: ρₕ ∈ [10, 5.000] Ω·m (log-uniforme), com camadas > 1000 Ω·m frequentes
- **Anisotropia TIV**: ρᵥ = ρₕ × fator ∈ [1.0, 3.0]
- **Espessuras**: 18 camadas internas, eₙ ∈ [0.5, 5.0] m (uniforme)
- Semente reprodutível: `seed = model_index`

Mede throughput absoluto de produção para geração de dataset de treinamento.


In [ ]:
# ── Célula 10: Geração dos 30k modelos ───────────────────────────────────────

N_MODELS_EXTRA = 30_000
N_LAYERS_EXTRA = 20
N_LAYERS_INTERNAL = N_LAYERS_EXTRA - 2  # = 18 espessuras

print(f"🔄 Gerando {N_MODELS_EXTRA:,} modelos aleatórios com {N_LAYERS_EXTRA} camadas...")
print(f"   Configuração: ρₕ ∈ [10, 5000] Ω·m | ρᵥ = ρₕ × [1.0, 3.0] | esp ∈ [0.5, 5.0] m")

t_gen_start = time.perf_counter()

# Pré-aloca arrays para todos os modelos
all_rho_h = np.zeros((N_MODELS_EXTRA, N_LAYERS_EXTRA), dtype=np.float64)
all_rho_v = np.zeros((N_MODELS_EXTRA, N_LAYERS_EXTRA), dtype=np.float64)
all_esp   = np.zeros((N_MODELS_EXTRA, N_LAYERS_INTERNAL), dtype=np.float64)

rng_master = np.random.RandomState(42)

for i in range(N_MODELS_EXTRA):
    # Seed individual para cada modelo (reprodutível)
    rng = np.random.RandomState(i)

    # ρₕ: log-uniforme em [10, 5000] Ω·m — distribui camadas condutivas e resistivas
    log_rho_h = rng.uniform(np.log(10.0), np.log(5000.0), size=N_LAYERS_EXTRA)
    rho_h = np.exp(log_rho_h)

    # Garantir que pelo menos 20% das camadas tenham ρₕ > 1000 Ω·m
    n_high = max(1, N_LAYERS_EXTRA // 5)
    idx_high = rng.choice(N_LAYERS_EXTRA, size=n_high, replace=False)
    rho_h[idx_high] = rng.uniform(1000.0, 5000.0, size=n_high)

    # ρᵥ: anisotropia TIV com fator entre 1.0 e 3.0 por camada
    aniso = rng.uniform(1.0, 3.0, size=N_LAYERS_EXTRA)
    rho_v = rho_h * aniso

    # Espessuras internas (não inclui semi-espaços)
    esp = rng.uniform(0.5, 5.0, size=N_LAYERS_INTERNAL)

    all_rho_h[i] = rho_h
    all_rho_v[i] = rho_v
    all_esp[i]   = esp

t_gen_end = time.perf_counter()

print(f"✅ {N_MODELS_EXTRA:,} modelos gerados em {t_gen_end - t_gen_start:.2f} s")
print(f"   ρₕ médio: {np.mean(all_rho_h):.1f} Ω·m")
print(f"   ρₕ max  : {np.max(all_rho_h):.1f} Ω·m")
print(f"   Fração com ρₕ > 1000 Ω·m: {np.mean(all_rho_h > 1000.0)*100:.1f}%")

# Posições fixas para todos os modelos (cobertura genérica)
POSITIONS_EXTRA = np.linspace(0.0, 50.0, N_POSITIONS)


In [ ]:
# ── Célula 11: Benchmark Numba — 30k modelos ──────────────────────────────────

print("🔄 Benchmark Numba JIT — 30.000 modelos (Config A: 1 freq, 1 TR, 0° dip)")
print("-" * 65)
print("   Aquecimento JIT (3 modelos)...")

# Aquecimento com os primeiros 3 modelos
for i in range(3):
    simulate(
        all_rho_h[i], all_rho_v[i], all_esp[i],
        POSITIONS_EXTRA, cfg=CFG_NUMBA_A,
    )

print("   Iniciando cronometragem...")
t_numba_start = time.perf_counter()

for i in range(N_MODELS_EXTRA):
    simulate(
        all_rho_h[i], all_rho_v[i], all_esp[i],
        POSITIONS_EXTRA, cfg=CFG_NUMBA_A,
    )
    if (i + 1) % 5000 == 0:
        elapsed = time.perf_counter() - t_numba_start
        rate = (i + 1) / elapsed * 3600
        print(
            f"   [{i+1:>6}/{N_MODELS_EXTRA}] "
            f"elapsed={elapsed:>6.1f}s | "
            f"throughput={rate:>9,.0f} mod/h"
        )

t_numba_total = time.perf_counter() - t_numba_start

NUMBA_30K_THROUGHPUT  = N_MODELS_EXTRA / t_numba_total * 3600
NUMBA_30K_MS_PER_MODEL = t_numba_total / N_MODELS_EXTRA * 1e3

print(f"\n✅ Numba JIT — {N_MODELS_EXTRA:,} modelos")
print(f"   Tempo total    : {t_numba_total:.2f} s ({t_numba_total/60:.1f} min)")
print(f"   Throughput     : {NUMBA_30K_THROUGHPUT:,.0f} mod/h")
print(f"   Tempo/modelo   : {NUMBA_30K_MS_PER_MODEL:.3f} ms")


In [ ]:
# ── Célula 12: Benchmark JAX — 30k modelos ────────────────────────────────────

if HAS_JAX:
    print(f"🔄 Benchmark JAX {JAX_BACKEND.upper()} — 30.000 modelos (Config A: 1 freq, 1 TR, 0° dip)")
    print("-" * 65)
    print("   Aquecimento JAX (3 modelos — inclui compilação XLA)...")
    clear_unified_jit_cache()

    for i in range(3):
        res = simulate_multi_jax(
            all_rho_h[i], all_rho_v[i], all_esp[i],
            POSITIONS_EXTRA,
            **CONFIG_A,
            cfg=CFG_JAX_UNIFIED,
        )
        jax.block_until_ready(res.H_tensor)

    xla_after_warmup = count_compiled_xla_programs()
    print(f"   XLA programs compilados após warmup: {xla_after_warmup}")
    print("   Iniciando cronometragem...")

    t_jax_start = time.perf_counter()

    for i in range(N_MODELS_EXTRA):
        res = simulate_multi_jax(
            all_rho_h[i], all_rho_v[i], all_esp[i],
            POSITIONS_EXTRA,
            **CONFIG_A,
            cfg=CFG_JAX_UNIFIED,
        )
        jax.block_until_ready(res.H_tensor)

        if (i + 1) % 5000 == 0:
            elapsed = time.perf_counter() - t_jax_start
            rate = (i + 1) / elapsed * 3600
            print(
                f"   [{i+1:>6}/{N_MODELS_EXTRA}] "
                f"elapsed={elapsed:>6.1f}s | "
                f"throughput={rate:>9,.0f} mod/h"
            )

    t_jax_total = time.perf_counter() - t_jax_start

    JAX_30K_THROUGHPUT   = N_MODELS_EXTRA / t_jax_total * 3600
    JAX_30K_MS_PER_MODEL = t_jax_total / N_MODELS_EXTRA * 1e3
    SPEEDUP_30K = t_numba_total / t_jax_total

    print(f"\n✅ JAX {JAX_BACKEND.upper()} — {N_MODELS_EXTRA:,} modelos")
    print(f"   Tempo total    : {t_jax_total:.2f} s ({t_jax_total/60:.1f} min)")
    print(f"   Throughput     : {JAX_30K_THROUGHPUT:,.0f} mod/h")
    print(f"   Tempo/modelo   : {JAX_30K_MS_PER_MODEL:.3f} ms")
    print(f"   XLA programs   : {count_compiled_xla_programs()}")

    print(f"\n{'='*60}")
    print(f"  SPEEDUP JAX {JAX_BACKEND.upper()} vs Numba JIT: {SPEEDUP_30K:.2f}×")
    print(f"  Numba : {NUMBA_30K_THROUGHPUT:>12,.0f} mod/h")
    print(f"  JAX   : {JAX_30K_THROUGHPUT:>12,.0f} mod/h")
    print(f"{'='*60}")
else:
    print("⚠️  JAX não disponível — benchmark JAX ignorado.")
    SPEEDUP_30K = None
    JAX_30K_THROUGHPUT = None


In [ ]:
# ── Célula 13: Tabela resumo experimento 30k ─────────────────────────────────

rows_30k = [
    {
        "Backend": "Numba JIT (CPU)",
        "Modelos": N_MODELS_EXTRA,
        "Camadas": N_LAYERS_EXTRA,
        "Tempo Total (s)": round(t_numba_total, 2),
        "Tempo Total (min)": round(t_numba_total / 60, 2),
        "ms/modelo": round(NUMBA_30K_MS_PER_MODEL, 3),
        "Throughput (mod/h)": int(NUMBA_30K_THROUGHPUT),
        "Speedup vs Numba": 1.0,
    },
]

if HAS_JAX and JAX_30K_THROUGHPUT is not None:
    rows_30k.append({
        "Backend": f"JAX {JAX_BACKEND.upper()} (unified)",
        "Modelos": N_MODELS_EXTRA,
        "Camadas": N_LAYERS_EXTRA,
        "Tempo Total (s)": round(t_jax_total, 2),
        "Tempo Total (min)": round(t_jax_total / 60, 2),
        "ms/modelo": round(JAX_30K_MS_PER_MODEL, 3),
        "Throughput (mod/h)": int(JAX_30K_THROUGHPUT),
        "Speedup vs Numba": round(SPEEDUP_30K, 2),
    })

df_30k = pd.DataFrame(rows_30k)
print("\n📊 Experimento Adicional — 30.000 Modelos Distintos")
print(f"   Configuração: Config A | {N_LAYERS_EXTRA} camadas | ρₕ ∈ [10, 5000] Ω·m | 600 posições")
print("=" * 90)
print(df_30k.to_string(index=False))


---
## Seção 5 — Plots dos Modelos Canônicos Testados

Para cada modelo: perfil de resistividade + resposta EM simulada (Config A e Config B).  
*(O experimento dos 30k modelos não inclui plots — apenas tabelas de throughput.)*


In [ ]:
# ── Célula 14: Pré-computar respostas para os plots ───────────────────────────

print("🔄 Computando respostas para os plots dos modelos canônicos...")

# Índices dos componentes do tensor H (9 componentes complex128):
# H = [Hxx, Hxy, Hxz, Hyx, Hyy, Hyz, Hzx, Hzy, Hzz] em complex128
# Convenção: eixo x = horizontal, z = axial (ao longo do poço)
IDX_HXX = 0   # Componente planar xx
IDX_HYY = 4   # Componente planar yy
IDX_HZZ = 8   # Componente axial zz
IDX_HXZ = 2   # Componente cruzada xz

# Armazena respostas para cada modelo
model_responses: Dict[str, Dict] = {}

for model_name in CANONICAL_MODELS:
    m = get_canonical_model(model_name)
    positions_z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, N_POSITIONS)

    # Config A — Numba (usada para os plots — mais estável e rápido)
    res_A = simulate(m.rho_h, m.rho_v, m.esp, positions_z, cfg=CFG_NUMBA_A)
    # H_tensor shape: (n_pos, nf, 9) — complex128
    # Seleciona frequência 0 (20 kHz)
    H_A = res_A.H_tensor[:, 0, :]  # (n_pos, 9)

    # Config B — Numba multi-freq/multi-dip
    res_B = simulate_multi_numba(
        m.rho_h, m.rho_v, m.esp, positions_z,
        **CONFIG_B, cfg=CFG_NUMBA_A,
    )
    # H_tensor shape: (nTR, nAngles, n_pos, nf, 9)
    # TR=0 (único), dip=0 => idx_dip=0, dip=30° => idx_dip=1
    H_B_dip0_f0  = res_B.H_tensor[0, 0, :, 0, :]  # 0°,  20 kHz, (n_pos, 9)
    H_B_dip0_f1  = res_B.H_tensor[0, 0, :, 1, :]  # 0°,  40 kHz
    H_B_dip30_f0 = res_B.H_tensor[0, 1, :, 0, :]  # 30°, 20 kHz
    H_B_dip30_f1 = res_B.H_tensor[0, 1, :, 1, :]  # 30°, 40 kHz

    # Posições de observação (z_obs depende do dip)
    z_obs_A  = positions_z
    z_obs_B0 = res_B.z_obs[0]   # dip=0°
    z_obs_B1 = res_B.z_obs[1]   # dip=30°

    model_responses[model_name] = {
        "model": m,
        "positions_z": positions_z,
        "z_obs_A": z_obs_A,
        "H_A": H_A,
        "z_obs_B0": z_obs_B0,
        "z_obs_B1": z_obs_B1,
        "H_B_dip0_f0": H_B_dip0_f0,
        "H_B_dip0_f1": H_B_dip0_f1,
        "H_B_dip30_f0": H_B_dip30_f0,
        "H_B_dip30_f1": H_B_dip30_f1,
    }
    print(f"  ✅ {model_name:<20} — Config A + Config B computadas")

print("\n✅ Todas as respostas computadas. Gerando plots...")


In [ ]:
# ── Célula 15: Plot — Oklahoma 3 camadas ──────────────────────────────────────

def plot_canonical_model(model_name: str, resp: Dict, config_label: str = ""):
    """Gera figura completa para um modelo canônico.

    Layout (3 colunas × 3 linhas):
      Coluna 0: Perfil de resistividade (ρₕ e ρᵥ)
      Coluna 1: Config A — Re(H) e Im(H) dos componentes principais
      Coluna 2: Config B — Comparação dips e frequências
    """
    m = resp["model"]
    positions_z = resp["positions_z"]
    H_A = resp["H_A"]
    H_B_d0_f0 = resp["H_B_dip0_f0"]
    H_B_d0_f1 = resp["H_B_dip0_f1"]
    H_B_d30_f0 = resp["H_B_dip30_f0"]
    H_B_d30_f1 = resp["H_B_dip30_f1"]
    z_obs_B0 = resp["z_obs_B0"]
    z_obs_B1 = resp["z_obs_B1"]

    # Interfaces do modelo
    interfaces = m.interfaces

    fig = plt.figure(figsize=(18, 11))
    fig.suptitle(
        f"{m.title} — Benchmark JAX GPU vs Numba JIT | "
        f"v1.6.0 | {m.anisotropy_type.upper()}",
        fontsize=13, fontweight="bold", y=1.01
    )

    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # ── Col 0: Perfil de resistividade (spanning 3 linhas) ────────────────────
    ax_rho = fig.add_subplot(gs[:, 0])
    depth_plot = np.concatenate([[-5.0], interfaces, [interfaces[-1] + 5.0]])
    rho_h_step = np.repeat(m.rho_h, 2)
    rho_v_step = np.repeat(m.rho_v, 2)
    depth_step = np.zeros(len(rho_h_step))
    depth_step[0::2] = depth_plot[:-1]
    depth_step[1::2] = depth_plot[1:]

    ax_rho.plot(rho_h_step, depth_step, "b-", linewidth=2.0, label=r"$\rho_h$")
    ax_rho.plot(rho_v_step, depth_step, "r--", linewidth=1.5, label=r"$\rho_v$")
    for iface in interfaces:
        ax_rho.axhline(iface, color="gray", linewidth=0.6, alpha=0.5, linestyle=":")
    ax_rho.axhline(positions_z[0],  color="green", linewidth=0.8, alpha=0.7, linestyle="-.",
                   label="range medição")
    ax_rho.axhline(positions_z[-1], color="green", linewidth=0.8, alpha=0.7, linestyle="-.")
    ax_rho.set_xscale("log")
    ax_rho.set_xlabel(r"Resistividade (Ω·m)")
    ax_rho.set_ylabel("Profundidade (m)")
    ax_rho.set_title(f"Perfil de Resistividade\n{m.n_layers} camadas | {m.reference}")
    ax_rho.invert_yaxis()
    ax_rho.legend(fontsize=9)
    ax_rho.grid(True, alpha=0.3)

    # ── Col 1: Config A — Hxx e Hzz (Re e Im) ────────────────────────────────
    ax_A_re = fig.add_subplot(gs[0, 1])
    ax_A_im = fig.add_subplot(gs[1, 1])
    ax_A_xz = fig.add_subplot(gs[2, 1])

    # Re(H)
    ax_A_re.plot(np.real(H_A[:, IDX_HXX]), positions_z, "b-",  lw=1.2, label=r"Re($H_{xx}$)")
    ax_A_re.plot(np.real(H_A[:, IDX_HZZ]), positions_z, "r-",  lw=1.2, label=r"Re($H_{zz}$)")
    ax_A_re.plot(np.real(H_A[:, IDX_HYY]), positions_z, "g--", lw=0.8, label=r"Re($H_{yy}$)", alpha=0.7)
    for iface in interfaces:
        ax_A_re.axhline(iface, color="gray", linewidth=0.5, alpha=0.4, linestyle=":")
    ax_A_re.set_title("Config A — Parte Real\n(20 kHz | TR=1m | dip=0°)")
    ax_A_re.set_xlabel("H (A/m)")
    ax_A_re.set_ylabel("Prof. (m)")
    ax_A_re.legend(fontsize=8)
    ax_A_re.invert_yaxis()
    ax_A_re.grid(True, alpha=0.25)

    # Im(H)
    ax_A_im.plot(np.imag(H_A[:, IDX_HXX]), positions_z, "b-",  lw=1.2, label=r"Im($H_{xx}$)")
    ax_A_im.plot(np.imag(H_A[:, IDX_HZZ]), positions_z, "r-",  lw=1.2, label=r"Im($H_{zz}$)")
    ax_A_im.plot(np.imag(H_A[:, IDX_HYY]), positions_z, "g--", lw=0.8, label=r"Im($H_{yy}$)", alpha=0.7)
    for iface in interfaces:
        ax_A_im.axhline(iface, color="gray", linewidth=0.5, alpha=0.4, linestyle=":")
    ax_A_im.set_title("Config A — Parte Imaginária")
    ax_A_im.set_xlabel("H (A/m)")
    ax_A_im.set_ylabel("Prof. (m)")
    ax_A_im.legend(fontsize=8)
    ax_A_im.invert_yaxis()
    ax_A_im.grid(True, alpha=0.25)

    # |H| (amplitude)
    ax_A_xz.plot(np.abs(H_A[:, IDX_HXX]), positions_z, "b-",  lw=1.2, label=r"$|H_{xx}|$")
    ax_A_xz.plot(np.abs(H_A[:, IDX_HZZ]), positions_z, "r-",  lw=1.2, label=r"$|H_{zz}|$")
    ax_A_xz.plot(np.abs(H_A[:, IDX_HXZ]), positions_z, "m--", lw=0.9, label=r"$|H_{xz}|$", alpha=0.8)
    ax_A_xz.set_xscale("log")
    for iface in interfaces:
        ax_A_xz.axhline(iface, color="gray", linewidth=0.5, alpha=0.4, linestyle=":")
    ax_A_xz.set_title("Config A — Amplitude |H|")
    ax_A_xz.set_xlabel("H (A/m)")
    ax_A_xz.set_ylabel("Prof. (m)")
    ax_A_xz.legend(fontsize=8)
    ax_A_xz.invert_yaxis()
    ax_A_xz.grid(True, alpha=0.25)

    # ── Col 2: Config B — comparação dips + freqs ─────────────────────────────
    ax_B_hxx = fig.add_subplot(gs[0, 2])
    ax_B_hzz = fig.add_subplot(gs[1, 2])
    ax_B_rat = fig.add_subplot(gs[2, 2])

    # Re(Hxx): dips e freqs
    ax_B_hxx.plot(np.real(H_B_d0_f0[:, IDX_HXX]),  z_obs_B0, "b-",  lw=1.2, label="0°, 20kHz")
    ax_B_hxx.plot(np.real(H_B_d0_f1[:, IDX_HXX]),  z_obs_B0, "b--", lw=1.0, label="0°, 40kHz", alpha=0.8)
    ax_B_hxx.plot(np.real(H_B_d30_f0[:, IDX_HXX]), z_obs_B1, "r-",  lw=1.2, label="30°, 20kHz")
    ax_B_hxx.plot(np.real(H_B_d30_f1[:, IDX_HXX]), z_obs_B1, "r--", lw=1.0, label="30°, 40kHz", alpha=0.8)
    for iface in interfaces:
        ax_B_hxx.axhline(iface, color="gray", linewidth=0.5, alpha=0.4, linestyle=":")
    ax_B_hxx.set_title(r"Config B — Re($H_{xx}$)")
    ax_B_hxx.set_xlabel("H (A/m)")
    ax_B_hxx.set_ylabel("Prof. (m)")
    ax_B_hxx.legend(fontsize=7)
    ax_B_hxx.invert_yaxis()
    ax_B_hxx.grid(True, alpha=0.25)

    # Re(Hzz): dips e freqs
    ax_B_hzz.plot(np.real(H_B_d0_f0[:, IDX_HZZ]),  z_obs_B0, "b-",  lw=1.2, label="0°, 20kHz")
    ax_B_hzz.plot(np.real(H_B_d0_f1[:, IDX_HZZ]),  z_obs_B0, "b--", lw=1.0, label="0°, 40kHz", alpha=0.8)
    ax_B_hzz.plot(np.real(H_B_d30_f0[:, IDX_HZZ]), z_obs_B1, "r-",  lw=1.2, label="30°, 20kHz")
    ax_B_hzz.plot(np.real(H_B_d30_f1[:, IDX_HZZ]), z_obs_B1, "r--", lw=1.0, label="30°, 40kHz", alpha=0.8)
    for iface in interfaces:
        ax_B_hzz.axhline(iface, color="gray", linewidth=0.5, alpha=0.4, linestyle=":")
    ax_B_hzz.set_title(r"Config B — Re($H_{zz}$)")
    ax_B_hzz.set_xlabel("H (A/m)")
    ax_B_hzz.set_ylabel("Prof. (m)")
    ax_B_hzz.legend(fontsize=7)
    ax_B_hzz.invert_yaxis()
    ax_B_hzz.grid(True, alpha=0.25)

    # Razão |Hzz/Hxx|: sensível à anisotropia e ao dip
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio_d0_f0  = np.abs(H_B_d0_f0[:, IDX_HZZ])  / (np.abs(H_B_d0_f0[:, IDX_HXX])  + 1e-30)
        ratio_d30_f0 = np.abs(H_B_d30_f0[:, IDX_HZZ]) / (np.abs(H_B_d30_f0[:, IDX_HXX]) + 1e-30)

    ax_B_rat.plot(ratio_d0_f0,  z_obs_B0, "b-",  lw=1.4, label=r"$|H_{zz}|/|H_{xx}|$ dip=0°")
    ax_B_rat.plot(ratio_d30_f0, z_obs_B1, "r-",  lw=1.4, label=r"$|H_{zz}|/|H_{xx}|$ dip=30°")
    ax_B_rat.axvline(2.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.7, label="ratio=2 (estático)")
    for iface in interfaces:
        ax_B_rat.axhline(iface, color="gray", linewidth=0.5, alpha=0.4, linestyle=":")
    ax_B_rat.set_title(r"Config B — Razão $|H_{zz}|/|H_{xx}|$")
    ax_B_rat.set_xlabel("Razão (adimensional)")
    ax_B_rat.set_ylabel("Prof. (m)")
    ax_B_rat.legend(fontsize=7)
    ax_B_rat.invert_yaxis()
    ax_B_rat.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.show()
    print(f"  Plot '{model_name}' gerado.\n")


# Gerar plot do primeiro modelo
print("=" * 70)
print("Plot 1/6 — Oklahoma 3 camadas")
print("=" * 70)
plot_canonical_model("oklahoma_3", model_responses["oklahoma_3"])


In [ ]:
# ── Célula 16: Plots — Oklahoma 5, Oklahoma 28 ────────────────────────────────

print("=" * 70)
print("Plot 2/6 — Oklahoma 5 camadas")
print("=" * 70)
plot_canonical_model("oklahoma_5", model_responses["oklahoma_5"])

print("=" * 70)
print("Plot 3/6 — Oklahoma 28 camadas (TIV forte — modelo mais complexo)")
print("=" * 70)
plot_canonical_model("oklahoma_28", model_responses["oklahoma_28"])


In [ ]:
# ── Célula 17: Plots — Devine 8, Hou 7, Viking Graben 10 ────────────────────

print("=" * 70)
print("Plot 4/6 — Devine 8 camadas (isotrópico)")
print("=" * 70)
plot_canonical_model("devine_8", model_responses["devine_8"])

print("=" * 70)
print("Plot 5/6 — Hou et al. (2006) 7 camadas")
print("=" * 70)
plot_canonical_model("hou_7", model_responses["hou_7"])

print("=" * 70)
print("Plot 6/6 — Viking Graben 10 camadas (reservatório Mar do Norte)")
print("=" * 70)
plot_canonical_model("viking_graben_10", model_responses["viking_graben_10"])


In [ ]:
# ── Célula 18: Relatório final consolidado ────────────────────────────────────

print("\n" + "="*80)
print("  RELATÓRIO FINAL — Benchmark JAX GPU vs Numba JIT")
print("  Geosteering AI v1.6.0 | Sprint 12")
print("="*80)

if HAS_JAX:
    print(f"\n  Dispositivo JAX: {JAX_BACKEND.upper()}")
    print(f"  Dispositivos   : {JAX_DEVICES}")

print("\n── Config A (1 freq, 1 TR, 0°, 600 pts) ──────────────────────────────")
df_A_numba_f = pd.DataFrame([
    {
        "Modelo": r.model_name, "n": r.n_layers,
        "Backend": r.backend,
        "ms/mod": round(r.time_median_s*1e3, 2),
        "Throughput(mod/h)": int(r.throughput_mh),
        "Speedup": round(r.speedup, 2),
    }
    for r in results_A
])
print(df_A_numba_f.to_string(index=False))

print("\n── Config B (2 freqs, 1 TR, 2 dips, 600 pts) ─────────────────────────")
df_B_f = pd.DataFrame([
    {
        "Modelo": r.model_name, "n": r.n_layers,
        "Backend": r.backend,
        "ms/mod": round(r.time_median_s*1e3, 2),
        "Throughput(mod/h)": int(r.throughput_mh),
        "Speedup": round(r.speedup, 2),
    }
    for r in results_B
])
print(df_B_f.to_string(index=False))

print("\n── Experimento 30k Modelos (Config A, 20 camadas) ─────────────────────")
print(f"  Numba JIT : {NUMBA_30K_THROUGHPUT:>10,.0f} mod/h | {NUMBA_30K_MS_PER_MODEL:.3f} ms/mod")
if HAS_JAX and JAX_30K_THROUGHPUT is not None:
    print(f"  JAX {JAX_BACKEND.upper():<5}  : {JAX_30K_THROUGHPUT:>10,.0f} mod/h | {JAX_30K_MS_PER_MODEL:.3f} ms/mod")
    print(f"  Speedup   : {SPEEDUP_30K:.2f}×")
    print(f"  {'✅ Meta ≥ 2× ATINGIDA' if SPEEDUP_30K >= 2.0 else '⚠️  Meta ≥ 2× NÃO atingida (revisar regressão 600×3)'}") 

print("\n" + "="*80)
print("  Benchmark concluído — Geosteering AI v1.6.0 Sprint 12")
print("="*80)
